# Dunnhumby seed 45 — 보완 M4와 새 M5 반복 확인
seed 43의 M5−M4 개선이 반복되는지 확인합니다. **새 M2 단독·상수-q·다른 시드는 학습하지 않습니다.**

- 모델과 학습 조건은 seed 43과 동일: N/V 선형 표현 W[p;q]+b, 공동학습, rho=0.05, ID64/축4/2층, L2=1e-3, lr=5e-4, batch8192, K=1 uniform, binary graph.
- M4: 1+0.5*q_C*(1-RBF_value_fit), 학습행 평균 정규화. 원형 M4가 아닙니다.
- 신규 상품 추천, MIN_ITEM_INTER=1, 개발 684–690. 최종 test·holdout 미사용.
- 최대 300 epoch, 25마다 평가. 100에서 대기를 초기화하고 이후 4회 연속 미개선이면 종료(최초 200). 전체 가격·구매금액 가중 적중값@10 최고 checkpoint 복원, 동률은 이른 epoch.
- M5−M4의 전체 가중 적중값@10·가중 NDCG@10 증가와 M1 대비 여섯 정확도 각각 99% 이상을 확인합니다. M1 대비 경제성과 통과는 별도입니다. @20/50·세그먼트·노출도 전부 저장합니다.
- 호환 M1 곡선은 재사용합니다. M1/M4 추가 학습은 계획표 확인 후 명시적으로 허용해야 합니다. 100 epoch 최종 결과만으로 조기 종료 기준모형을 대체하지 않습니다.
- 새 M5 최대 300 model-epoch + 누락 기준모형당 최대 300. 실행 전에 비용을 확인하세요.
- 기존 seed 43 노트북·캐시는 그대로 보존합니다. 이 노트북은 별도 고정 코드·결과 폴더를 사용합니다.
- 1–3번은 학습 없음, 4번부터 학습. seed 44 및 H&M과 별도 GPU 런타임 사용. 매 epoch 저장하며 재실행 시 이어집니다. 세션 자동 재연결은 아닙니다.
- 단일 추가 개발 시드이며 유의성·일반화·CLV 고유효과를 주장하지 않습니다. seed 44 결과와 무관하게 불리한 결과도 seed 43·44·45 모두 함께 보고합니다.


In [ ]:
# 1. 고정 코드 준비
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json
from pathlib import Path
SOURCE_COMMIT = '5533e3e2d3dd527fd8aef69b364d11c4b06abe76'
REPO = Path('/content/clv-linear-nv-s45-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '별도 런타임 또는 세션 재시작이 필요합니다.'
os.chdir(REPO)
sys.path.insert(0,str(REPO))


In [ ]:
# 2. seed 45, M5만 신규 후보로 설정
import torch
import pandas as pd
import lightgcn_clv_history_linear_nv_early_stop as screen
cfg = screen.configure(seeds=(45,), include_m2=False, include_controls=False,
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_history_linear_nv_es_seed45_v2')
assert [s['model_id'] for s in screen.specs(cfg)] == ['m5_linear_nv']
print(json.dumps(screen.preflight(cfg),ensure_ascii=False,indent=2))
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요.'


In [ ]:
# 3. 재사용/재개/추가 학습 계획만 확인 — 학습 없음
prepared = screen.prepare(cfg)
plan = pd.DataFrame(prepared['plan'])
display(plan)
print('최대 추가 model-epoch:', int(plan.max_additional_epochs.sum()))
required = [f"{p['seed']}:{p['model_id']}" for p in prepared['plan']
    if p['model_id'] in ('m1','m4') and p['action']=='train_from_start']
print('확인 후 다음 셀에 허용할 기준모형:',required)


In [ ]:
# 4. 실제 학습. 3번에서 M1 재사용 / M4 추가학습이면 ['45:m4']로 변경하세요.
# M1도 미발견이면 추가 비용을 먼저 확인하세요. 빈 목록에서는 필요한 승인이 없을 때 학습 전 중단합니다.
APPROVED_BASELINE_FITS = []
absolute = screen.run(cfg,prepared=prepared,approved_baseline_fits=APPROVED_BASELINE_FITS)
paths = absolute.attrs['paths']
print(json.dumps(paths,ensure_ascii=False,indent=2))


In [ ]:
# 5. 선택 epoch 및 판정 — 전체 원본은 ZIP에 포함
display(absolute[['seed','model_id','selected_epoch','stopped_epoch','stop_reason']])
summary = pd.read_csv(paths['summary'])
metrics = list(screen.base.ACCURACY)+list(screen.fixed.PRIMARY)+[
    'price_purchase_amount_weighted_hit@20','price_purchase_amount_weighted_hit@50','vndcg@20','vndcg@50']
print(summary[summary.metric.isin(metrics)].to_string(index=False))
display(pd.read_csv(paths['reading']))
print('directional_pass = M5-M4 개선 + M1 정확도 보호. overall_goal_direction = M1 대비 경제성과까지 개선.')
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path('/content/m5_linear_nv_seed45_results.zip')
with ZipFile(archive,'w',compression=ZIP_DEFLATED) as z:
    for path in paths.values():
        z.write(path,arcname=Path(path).name)
files.download(str(archive))
